## Introduction to DataIngestion

In [1]:
import langchain
from typing import List,Dict,Any
import pandas as pd

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

f:\D Drive\Udemy_RAG (Krish Naik)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Understanding the Document Structure in Langchain

In [3]:
## Create a Simple Document
doc = Document(
    page_content="This is the Main Text Content that will be embedded and searched",
    metadata = {
        "source" : "example.txt",
        "page" : 1,
        "author" : "Krish Naik",
        "date_created" : "2024-06-08",
        "custom_field": "Any Value"
    }
)
print("Document Structure")
print(f" Content: {doc.page_content}")
print(f" MetaData: {doc.metadata}")

print("\n📄 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")


Document Structure
 Content: This is the Main Text Content that will be embedded and searched
 MetaData: {'source': 'example.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2024-06-08', 'custom_field': 'Any Value'}

📄 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [4]:
type(doc)

langchain_core.documents.base.Document

### Text Files (.txt) - THe Simplest Case(#3-text-files)

In [5]:
## Create a Simple Txt File
import os
os.makedirs("data/text_files",exist_ok = True)

In [6]:

sample_text = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems."""
}

In [7]:
for filePath,content in sample_text.items():
    with open(filePath,'w',encoding = "utf-8") as f:
        f.write(content)

### Text Loader - Read Single File

In [8]:
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader = TextLoader("data/text_files/python_intro.txt",encoding = "utf-8")

C:\Users\hp\AppData\Local\Temp\ipykernel_35112\1851300596.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [9]:
loader

In [10]:
documents = loader.load()

print(f"📄 Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

📄 Loaded 1 document
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Metadata: {'source': 'data/text_files/python_intro.txt'}


### Directory Loader- Multiple Text Files

In [11]:
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern that Match the text extension
    loader_cls= TextLoader, ## Loader Class to Use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress= True
)

documents = dir_loader.load()
print(f" Loaded {len(documents)} Documents")
for i,doc in enumerate(documents):
    print(f"\n Document {i+1}")
    print(f" Source: {doc.metadata['source']}")
    print(f" Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 809.87it/s]

 Loaded 2 Documents

 Document 1
 Source: data\text_files\machine_learning.txt
 Length: 569 characters

 Document 2
 Source: data\text_files\python_intro.txt
 Length: 489 characters


In [12]:

print("\n📁 DirectoryLoader Characteristics:")
print("✅ Advantages:")
print(" - Loads multiple files at once")
print(" - Supports glob patterns")
print(" - Progress tracking")
print(" - Recursive directory scanning")

print("\n❌ Disadvantages:")
print(" - All files must be same type")
print(" - Limited error handling per file")
print(" - Can be memory intensive for large directories")


📁 DirectoryLoader Characteristics:
✅ Advantages:
 - Loads multiple files at once
 - Supports glob patterns
 - Progress tracking
 - Recursive directory scanning

❌ Disadvantages:
 - All files must be same type
 - Limited error handling per file
 - Can be memory intensive for large directories


## Text Splitting Stratergies

In [14]:
### Different Text Splitters
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems.'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming l

In [15]:
### Method-1 : Character Text Splitters
text = documents[0].page_content
print(text)

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems.


In [17]:
### Method-1 : Character Based Splitting
print(" Character Text Splitter")
char_splitter = CharacterTextSplitter(
    separator="\n", # Split on New Lines
    chunk_size = 100, # Max Chunk Size in Characters
    chunk_overlap = 20, # Overlap Between Chunks
    length_function = len # How to measure chunk_size
)
char_chunks = char_splitter.split_text(
    text=text
)

print(f" Created {len(char_chunks)} chunks")
print(f"First Chunk: {char_chunks[0][:100]} ...")

 Character Text Splitter
 Created 8 chunks
First Chunk: Machine Learning Basics ...


In [18]:
print(char_chunks)

['Machine Learning Basics', 'Machine learning is a subset of artificial intelligence that enables systems to learn and improve', 'from experience without being explicitly programmed. It focuses on developing computer programs', 'that can access data and use it to learn for themselves.\nTypes of Machine Learning:', '1. Supervised Learning: Learning with labeled data', '2. Unsupervised Learning: Finding patterns in unlabeled data', '3. Reinforcement Learning: Learning through rewards and penalties', 'Applications include image recognition, speech processing, and recommendation systems.']


In [ ]:
### Method-2 : Recursive Character Text Splitter (Mostly Used)
print("Recursive Character Text Splitter")
recursiveSplitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n",""," "], ## Try These Separators in order
    chunk_size = 200,
    chunk_overlap = 20,
    length_function = len
)

recursive_chunks = recursiveSplitter.split_text(text)
print(f"Created {len(recursive_chunks)}  chunks")
print(f" First Chunk: {recursive_chunks[0][:100]} ...")

Recursive Character Text Splitter
Created 6  chunks
 First Chunk: Machine Learning Basics ...


In [21]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])

Machine Learning Basics
-----------------
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs


In [23]:
## Method-3: Token-Based Splitting
print("Token Based Splitting")
token_splitter = TokenTextSplitter(
    chunk_size = 20, # Size in token (text Characters)
    chunk_overlap = 10
)
token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)}  chunks")
print(f" First Chunk: {token_chunks[0][:100]} ...")

Token Based Splitting
Created 10  chunks
 First Chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system ...


In [24]:
# 📊 Comparison
print("\n🔹 Text Splitting Methods Comparison:")

print("\nCharacterTextSplitter:")
print("    ✅ Simple and predictable")
print("    ✅ Good for structured text")
print("    ❌ May break mid-sentence")
print("    Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("    ✅ Respects text structure")
print("    ✅ Tries multiple separators")
print("    ✅ Best general-purpose splitter")
print("    ❌ Slightly more complex")
print("    Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("    ✅ Respects model token limits")
print("    ✅ More accurate for embeddings")
print("    ❌ Slower than character-based")
print("    Use when: Working with token-limited models")


🔹 Text Splitting Methods Comparison:

CharacterTextSplitter:
    ✅ Simple and predictable
    ✅ Good for structured text
    ❌ May break mid-sentence
    Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
    ✅ Respects text structure
    ✅ Tries multiple separators
    ✅ Best general-purpose splitter
    ❌ Slightly more complex
    Use when: Default choice for most texts

TokenTextSplitter:
    ✅ Respects model token limits
    ✅ More accurate for embeddings
    ❌ Slower than character-based
    Use when: Working with token-limited models
